In [ ]:
import os, sys, glob
_found = sorted(glob.glob("/kaggle/input/**/pipeline/decoder.py", recursive=True))
REPO = os.environ.get("MTAH_REPO") or next((p for p in ["/kaggle/working/Hackaton_Barcellona", os.path.abspath("..")] + [os.path.dirname(os.path.dirname(f)) for f in _found] if os.path.isdir(os.path.join(p, "pipeline"))), None)
assert REPO, "Imposta MTAH_REPO alla cartella della repo"
sys.path[:0] = [REPO, os.path.join(REPO, "pipeline"), os.path.join(REPO, "starter"), os.path.join(REPO, "evaluation")]

# C1 · Rete sui giocatori: il possesso anche quando la palla non si vede

**Su Kaggle: Settings → Accelerator → GPU T4.** Senza GPU il notebook funziona lo stesso, ma la
griglia completa diventa troppo lenta (usare `QUICK = True`).

Sulla simulazione DFL della nostra catena il costo più grande sono i **vuoti della nostra palla**:
−0,107 di spellF1 (CONTESTO.md, sezione 6). Dove la palla manca non c'è tocco né giocatore più
vicino, e il decoder A1 resta senza indizi su H/A. Ma i giocatori si muovono comunque come la
squadra che ha la palla: chi attacca sale e si allarga, chi difende arretra e stringe sul
portatore. Questo notebook addestra una piccola rete che legge quei movimenti.

Tutta la logica è in `pipeline/player_net.py`; qui si eseguono i passi e si leggono i risultati.
Le partite DFL sono altre partite (CC BY 4.0): addestrare qui è permesso. Si stampano solo aggregati.

## 1. Perché una rete sui giocatori, e com'è fatta

**Riferimenti.**

- **Borghesi et al. 2023**: una TCN per ogni oggetto (giocatori e palla) e un'aggregazione che non
  dipende dall'ordine degli oggetti. Possesso all'86,2% su tracking ottico. Nell'ablation, la
  sola palla dà circa il 50%, palla più squadra degli oggetti circa l'85%.
- **Kim et al. 2023, "Ball Radar"** (Set Transformer + Bi-LSTM): possesso di squadra all'85% con
  la palla **del tutto nascosta**, al 97% con l'80% nascosta. La Bi-LSTM fa meglio del Transformer.
- **PathCRF 2026**: lavora sullo stesso dataset DFL.

Il codice pubblico di Ball Radar (github.com/pientist/ballradar) è solo un riferimento: qui è
tutto riscritto da zero.

**Perché un modello che non dipende dall'ordine (permutation-invariant).** Il nostro tracking
reale non ha identità: gli slot sono frammenti di traccia, numerati a caso, che vanno e vengono.
Ogni oggetto passa quindi nello **stesso** encoder. Poi si fa una media pesata con attenzione,
separata per casa, ospiti e palla. Scambiare due slot non cambia il risultato, e lo verifica un
test.

**Ingressi.** Per ogni oggetto e per ogni passo della finestra:

- posizione in metri, ruotata in modo che la squadra di casa attacchi verso +x (la direzione si
  ricava dai baricentri delle squadre all'inizio di ogni tempo, senza etichette);
- velocità e modulo della velocità;
- squadra (one-hot casa / ospiti / palla), flag "palla di Metrica", flag "mancante";
- posizione rispetto alla palla di riferimento e distanza.

**Due palle.** La palla è un oggetto come gli altri. Ce ne sono due: la nostra (con i vuoti) e
quella tipo Metrica (sfocata, assente a gioco fermo). Quando una palla non c'è, ha il flag
"mancante" acceso.

**Architettura.**

1. TCN condivisa: 5 blocchi residui con dilatazioni 1–16, che coprono tutta la finestra.
2. Embedding al passo centrale.
3. Un passo di interazione: ogni oggetto vede la media di tutti.
4. Pooling con attenzione per gruppo.
5. MLP finale: 2 logit (H/A) più 1 logit ausiliario, "c'è un cambio di possesso entro ±0,5 s".

Il task ausiliario aiuta la rete a capire dove avvengono i passaggi di consegne.

## 2. Finestra e dimensione: iperparametri, non scelte fisse

| Iperparametro | Valori | Da dove vengono |
|---|---|---|
| Finestra W | **100 frame (4 s)** | Borghesi et al. 2023; il contesto ±4 s di Lang et al. 2022 |
| | **250 frame (10 s)** | Kim et al. 2023 e PathCRF |
| Canali della TCN | **32, 64, 128** | ~30k, ~110k, ~400k parametri |

La finestra è sempre campionata in 51 passi: ogni 2 frame con W = 100, ogni 5 con W = 250. Le
velocità sono calcolate a 25 fps prima di campionare, quindi non si perdono.

**Perché una rete piccola.** Abbiamo solo 7 partite, cioè poche migliaia di possessi. Una rete
grande imparerebbe a memoria le partite di training. Questa scelta va verificata con il
confronto qui sotto, non presa per buona.

**Regola di scelta, fissata prima di guardare i numeri.** Vince la configurazione con la miglior
accuratezza media **nel vuoto di palla** sui fold. Se altre sono entro 0,5 punti, vince la più
piccola.

**Protocollo leave-one-match-out.** Per ogni fold:

- 1 partita di test;
- 1 partita di validazione per l'early stopping;
- 5 partite di training.

Il modello finale si addestra su tutte e 7 le partite, per il numero mediano di epoche migliori
dei fold.

**Mascheramento della palla in training.** Oltre ai vuoti simulati, la nostra palla viene
nascosta così:

- a blocchi, con lunghezze fisse (`dfl_sim.MASK_GAPS`), fino a ×3, su metà degli esempi;
- per tutta la finestra nel 15% degli esempi;
- insieme alla palla di Metrica, cioè senza nessuna palla, nel 10%.

Così la rete impara anche a fare a meno della palla.

**Condizioni.** Sono quelle di `pipeline/dfl_sim.py`, le stesse di `decoder.simulate`:

- giocatori con rumore, buchi, tracce che cambiano slot e 1% di squadra sbagliata;
- una sola palla, quella fornita, che manca anche a gioco vivo. Nel codice occupa il primo
  posto palla ("nostra"); il secondo resta vuoto.

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import dfl_paths, decoder, player_net as pn

QUICK = False          # True: 2 fold, 3 epoche, solo W=100 e 64 canali (pochi minuti)
pd.set_option("display.width", 200)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nessuna")
print("partite DFL:", len(dfl_paths.match_paths()), "in", dfl_paths.converted_dir())
import dfl_sim
print("simulazione:", dfl_sim.level(), "| output:", dfl_paths.out_dir(), "| modelli:", pn.models_dir())
for w in pn.GRID_W:
    for ch in pn.GRID_CH:
        print("  W={:<3} canali={:<3} parametri {:>7,}".format(w, ch, pn.n_params(dict(pn.DEFAULTS, W=w, channels=ch))))

## 3. Test su dati sintetici

I test girano in pochi minuti su CPU, su partite finte in una cartella temporanea. Verificano:

- che le estrazioni casuali coincidano con quelle di `decoder.simulate`;
- la direzione d'attacco;
- che l'ordine degli slot non conti;
- il canale della rete nel decoder;
- la catena completa: training, scelta, `_net.npz`, modello salvato e ricaricato, lettura da A1.

In [ ]:
import test_player_net
for name in sorted(n for n in dir(test_player_net) if n.startswith("test_")):
    getattr(test_player_net, name)()
    print("ok", name)

## 4. Griglia leave-one-match-out, scelta, uscite

Con la GPU T4 la griglia completa (2 finestre × 3 dimensioni × 7 fold) richiede circa
45–75 minuti; una sola configurazione circa 10. Ogni fold stampa la sua accuratezza.

Alla fine la cella:

- scrive `<match>_net.npz` (out-of-fold, configurazione scelta) in `dfl_outputs`;
- addestra il modello finale e lo salva in `models/player_net.pt` + `player_net.json`.

Per ridurre i tempi si può passare `grid_w=(100,)` oppure `grid_ch=(32, 64)`.

In [ ]:
res = pn.run(quick=QUICK)
print("dispositivo:", res["device"], "| simulazione:", res["sim"], "| scelta:", res["chosen"])

## 5. Come leggere le metriche

Le accuratezze sono calcolate sui frame vivi con possesso, frame per frame:

- `acc_seen`: frame con la nostra palla visibile;
- `acc_gap`: frame con la nostra palla **nel vuoto**. È il numero chiave.

Il confronto `near_*` è il giocatore più vicino alla palla della catena A1, sugli stessi frame:
nostra palla, vuoti fino a 13 frame interpolati, Metrica oltre. È l'indizio H/A che il decoder ha
oggi. Un "più vicino" assente o un P(home) = 0,5 contano mezzo punto, come una moneta.

Riferimenti per leggere i numeri:

- Borghesi et al. arrivano all'86% con la palla;
- Kim et al. all'85% senza palla.

Se `acc_gap` supera nettamente `near_gap`, la rete porta informazione proprio dove manca.

In [ ]:
print("Accuratezza media sui fold, per configurazione (params = numero di parametri):")
display(res["grid"][["params", "acc_all", "acc_seen", "acc_gap", "acc_gap_min", "near_all", "near_seen", "near_gap", "gap_share", "best_epoch"]])
print("configurazione scelta:", res["chosen"])

In [ ]:
print("Per partita, configurazione scelta:")
pm_ = res["per_match"]
display(pm_[pm_.cfg == res["chosen"]][["fold", "params", "best_epoch", "frames", "gap_share", "acc_all", "acc_seen", "acc_gap", "near_all", "near_gap"]].round(3).reset_index(drop=True))
print("Senza nessuna palla (né la nostra né Metrica), stessa rete:")
display(res["ablation"].drop(columns="match"))

## 6. Effetto a valle: spellF1 del decoder A1 con e senza la rete

Il decoder A1 somma in log-odds tutti gli indizi su H/A: giocatore più vicino, tocchi, regola, e
B2 se c'è. Con il file `<match>_net.npz` aggiunge anche `p_home` della rete, con peso
`decoder.NET_WEIGHT` (1,0).

A differenza degli altri canali, quello della rete resta acceso **anche dove la palla manca**.

I parametri del decoder si scelgono su 6 partite e si misurano sulla settima, come in A1. Sotto
circa 0,04 di spellF1 le differenze sono rumore.

In [ ]:
display(res["downstream"])

## 7. Un tratto di partita

La banda in alto è la verità DFL; la linea è P(home) della rete (out-of-fold); in grigio i tratti
in cui la nostra palla è nel vuoto. Il tratto scelto è il minuto e mezzo con più frame vivi nel
vuoto.

In [ ]:
pn.plot_example(res, seconds=90)
plt.show()

## 8. Uscite e cosa riportare al team

**Uscite.**

- `/kaggle/working/dfl_outputs/<match>_net.npz` (`p_home`, `variant`): A1 lo legge da solo se gira
  dopo, nella stessa sessione.
- `/kaggle/working/models/player_net.pt` + `player_net.json`: **da scaricare**. Servono per
  applicare la rete a BAR-ATH in locale con `player_net.load_model()` e
  `player_net.predict_p_home(...)`, su CPU.

**Cosa riportare.**

- la tabella della griglia: accuratezza vista / nel vuoto e parametri per ogni configurazione,
  più la configurazione scelta;
- `acc_gap` contro `near_gap`, e l'accuratezza senza nessuna palla;
- lo spellF1 di A1 con e senza rete, e il livello della simulazione.

Solo numeri aggregati su DFL. Niente dati o output della partita BAR-ATH in strumenti AI online.

In [ ]:
# Facoltativo: rieseguire A1 completo, che ora legge anche i _net.npz (vedi la riga "sources").
RUN_A1 = False
if RUN_A1:
    r = decoder.evaluate_dfl()
    print(r["sources"])